# Predicción de Jugadas en Póker Texas Hold'em con IA
## Comparativa: Minimax · Red Bayesiana · Cadenas de Markov · TD Learning

---

## Tabla de Contenidos

- [Sección 0: Instalación y Configuración](#seccion-0)
- [Sección 1: Fundamentos — Representación del Juego](#seccion-1)
- [Sección 2: Modelo 1 — Minimax con Poda Alpha-Beta](#seccion-2)
- [Sección 3: Modelo 2 — Red Bayesiana](#seccion-3)
- [Sección 4: Modelo 3 — Cadenas de Markov](#seccion-4)
- [Sección 5: Modelo 4 — TD Learning (Q-Learning)](#seccion-5)
- [Sección 6: Modelo Combinado (Ensemble)](#seccion-6)
- [Sección 7: Comparación y Evaluación](#seccion-7)

<a id='seccion-0'></a>
## Sección 0: Instalación y Configuración

Instalamos todas las dependencias necesarias y configuramos el entorno de ejecución con una semilla aleatoria fija para garantizar reproducibilidad.


In [1]:
# Importaciones globales
import random
import time
import warnings
import itertools
from copy import deepcopy
from typing import List, Tuple, Dict, Optional, Any
from enum import Enum, auto
from collections import defaultdict

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm import tqdm

# Evaluador de manos de póker
from treys import Card, Evaluator, Deck

# Red Bayesiana
from pgmpy.models import DiscreteBayesianNetwork as BayesianNetwork
from pgmpy.factors.discrete import TabularCPD
from pgmpy.inference import VariableElimination
from pgmpy.estimators import MaximumLikelihoodEstimator

warnings.filterwarnings('ignore')

# Semilla global
RANDOM_STATE = 42
random.seed(RANDOM_STATE)
np.random.seed(RANDOM_STATE)

print("Entorno configurado correctamente.")
print(f"   Semilla aleatoria global: {RANDOM_STATE}")


c:\Users\gabri\AppData\Local\Programs\Python\Python312\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
c:\Users\gabri\AppData\Local\Programs\Python\Python312\Lib\site-packages\pgmpy\estimators\__init__.py:4: FutureWarning: `pgmpy.estimators.StructureScore` is deprecated and will be removed in v1.3.0. Use `pgmpy.structure_score` instead.
  from .StructureScore import (


Entorno configurado correctamente.
   Semilla aleatoria global: 42


<a id='seccion-1'></a>
## Sección 1: Fundamentos - Representación del Juego

Antes de implementar cualquier modelo de IA, necesitamos una representación uniforme del juego. Todos los agentes operarán sobre la misma clase `GameState`, lo que garantiza una comparación justa.

### Componentes del entorno:
1. **Mazo y cartas**: representadas en formato compatible con `treys`
2. **Estado del juego**: encapsula toda la información necesaria para tomar una decisión
3. **Espacio de acciones**: FOLD, CHECK, CALL, RAISE (3 tamaños), ALL_IN
4. **Evaluación de manos**: usando `treys` + estimación de equity por Monte Carlo


In [2]:
# 1.1  Enumeraciones y constantes del juego

class Phase(Enum): # Fases del juego Texas Hold'em
    PREFLOP  = 0
    FLOP     = 1
    TURN     = 2
    RIVER    = 3
    SHOWDOWN = 4


class Action(Enum): # Acciones disponibles para un jugador
    FOLD        = 0
    CHECK       = 1
    CALL        = 2
    RAISE_HALF  = 3   # 0.5× pot
    RAISE_POT   = 4   # 1×  pot
    RAISE_2POT  = 5   # 2×  pot
    ALL_IN      = 6


# Mapeo legible de acción → descripción
ACTION_LABELS = {
    Action.FOLD:       "Fold",
    Action.CHECK:      "Check",
    Action.CALL:       "Call",
    Action.RAISE_HALF: "Raise ½ pot",
    Action.RAISE_POT:  "Raise 1x pot",
    Action.RAISE_2POT: "Raise 2x pot",
    Action.ALL_IN:     "All-in",
}

# Score numérico para el ensemble (agresividad creciente)
ACTION_SCORES = {
    Action.FOLD:       0,
    Action.CHECK:      1,
    Action.CALL:       2,
    Action.RAISE_HALF: 3,
    Action.RAISE_POT:  4,
    Action.RAISE_2POT: 5,
    Action.ALL_IN:     6,
}

STARTING_STACK = 1000   # fichas iniciales por jugador
BIG_BLIND      = 20     # ciega grande

print("Enumeraciones y constantes definidas.")


Enumeraciones y constantes definidas.


In [3]:
# 1.2  Clase GameState

class GameState:
    """
    Encapsula el estado completo de una mano de Texas Hold'em.

    - Attributes
    hole_cards : List[int]
        Las dos cartas privadas del jugador (formato treys).
    community_cards : List[int]
        Cartas comunitarias visibles (0-5 cartas).
    pot : float
        Fichas acumuladas en el bote.
    stacks : List[float]
        Stack de cada jugador [jugador_0, jugador_1].
    current_bet : float
        Apuesta actual que hay que igualar.
    phase : Phase
        Fase del juego (PREFLOP, FLOP, TURN, RIVER, SHOWDOWN).
    acting_player : int
        Índice del jugador que debe actuar (0 o 1).
    action_history : List[Tuple]
        Historial de acciones (fase, jugador, acción).
    position : int
        Posición del jugador principal (0=EARLY/SB, 1=LATE/BB).
    """

    def __init__(
        self,
        hole_cards: List[int],
        community_cards: Optional[List[int]] = None,
        pot: float = 0.0,
        stacks: Optional[List[float]] = None,
        current_bet: float = 0.0,
        phase: Phase = Phase.PREFLOP,
        acting_player: int = 0,
        position: int = 0,
    ):
        self.hole_cards       = hole_cards
        self.community_cards  = community_cards if community_cards is not None else []
        self.pot              = pot
        self.stacks           = stacks if stacks is not None else [STARTING_STACK, STARTING_STACK]
        self.current_bet      = current_bet
        self.phase            = phase
        self.acting_player    = acting_player
        self.action_history   = []
        self.position         = position

    # Helpers

    @property
    def is_terminal(self) -> bool:
        """True si la mano ha terminado."""
        return self.phase == Phase.SHOWDOWN or any(s <= 0 for s in self.stacks)

    @property
    def pot_odds(self) -> float:
        """Relación call_amount / (pot + call_amount). 0 si no hay apuesta."""
        if self.current_bet <= 0:
            return 0.0
        call_cost = self.current_bet
        return call_cost / (self.pot + call_cost)

    @property
    def stack_ratio(self) -> float:
        """Stack del jugador activo sobre el stack total."""
        total = sum(self.stacks)
        return self.stacks[self.acting_player] / total if total > 0 else 0.5

    def __repr__(self) -> str:
        cards_str = Card.ints_to_pretty_str(self.hole_cards) if self.hole_cards else "[]"
        comm_str  = Card.ints_to_pretty_str(self.community_cards) if self.community_cards else "[]"
        return (
            f"GameState(phase={self.phase.name}, "
            f"hole={cards_str}, community={comm_str}, "
            f"pot={self.pot:.0f}, bet={self.current_bet:.0f}, "
            f"stacks={[int(s) for s in self.stacks]})"
        )


print("GameState definido.")


GameState definido.


In [4]:
# 1.3  Acciones disponibles

def get_legal_actions(state: GameState) -> List[Action]:
    """
    Retorna la lista de acciones legales en el estado actual.

    Reglas:
    - FOLD siempre es legal.
    - CHECK es legal si no hay apuesta pendiente.
    - CALL es legal si hay apuesta pendiente y el jugador tiene fichas.
    - RAISE es legal si el jugador tiene suficientes fichas.
    - ALL_IN siempre es legal si el jugador tiene fichas.

    Parameters, state : GameState

    Returns, List[Action]
    """
    stack   = state.stacks[state.acting_player]
    bet     = state.current_bet
    pot     = state.pot
    actions = [Action.FOLD]

    if bet <= 0:
        actions.append(Action.CHECK)
    else:
        if stack >= bet:
            actions.append(Action.CALL)

    # Raises
    for action, multiplier in [
        (Action.RAISE_HALF, 0.5),
        (Action.RAISE_POT,  1.0),
        (Action.RAISE_2POT, 2.0),
    ]:
        raise_amount = max(multiplier * pot, BIG_BLIND)
        if stack >= bet + raise_amount:
            actions.append(action)

    # ALL_IN
    if stack > 0:
        actions.append(Action.ALL_IN)

    return actions


def action_to_bet_size(action: Action, state: GameState) -> float: # Convierte una acción a su tamaño de apuesta en fichas
    pot = state.pot
    bet = state.current_bet
    if action == Action.FOLD:        return 0.0
    if action == Action.CHECK:       return 0.0
    if action == Action.CALL:        return bet
    if action == Action.RAISE_HALF:  return bet + max(0.5 * pot, BIG_BLIND)
    if action == Action.RAISE_POT:   return bet + max(1.0 * pot, BIG_BLIND)
    if action == Action.RAISE_2POT:  return bet + max(2.0 * pot, BIG_BLIND)
    if action == Action.ALL_IN:      return state.stacks[state.acting_player]
    return 0.0


print("Espacio de acciones definido.")
print("  Ejemplo de acciones legales en estado vacío:")
demo_state = GameState(hole_cards=[], stacks=[500, 500], current_bet=0, pot=100)
print(" ", [a.name for a in get_legal_actions(demo_state)])


Espacio de acciones definido.
  Ejemplo de acciones legales en estado vacío:
  ['FOLD', 'CHECK', 'RAISE_HALF', 'RAISE_POT', 'RAISE_2POT', 'ALL_IN']


In [5]:
# 1.4  Evaluación de manos y cálculo de equity

EVALUATOR = Evaluator()

# Mazo estándar en formato treys
ALL_CARDS = [Card.new(r + s) for r in "23456789TJQKA" for s in "shdc"]


def normalize_hand_strength(score: int) -> float:
    """
    Normaliza el score de treys a [0, 1].

    treys usa 1 = mejor mano (Royal Flush), 7462 = peor (7-high).
    Invertimos y normalizamos para que 1.0 = la mano más fuerte.

    Parameters, score : int  (1 ≤ score ≤ 7462)

    Returns, float  strength ∈ [0, 1]
    """
    return 1.0 - (score - 1) / 7461.0


def evaluate_hand(hole_cards: List[int], community_cards: List[int]) -> float:
    """
    Evalúa la fuerza normalizada de una mano (0→peor, 1→mejor).
    Requiere al menos 3 cartas comunitarias (flop).
    """
    if len(community_cards) < 3:
        return 0.5   # Sin información suficiente → fuerza neutra
    score = EVALUATOR.evaluate(community_cards, hole_cards)
    return normalize_hand_strength(score)


def simulate_equity(
    hole_cards: List[int],
    community_cards: List[int],
    n_simulations: int = 1000,
) -> float:
    """
    Estima la equity del jugador mediante simulación Monte Carlo.

    Para cada simulación:
    1. Completa el tablero con cartas aleatorias del mazo restante.
    2. Sortea 2 cartas aleatorias para el oponente.
    3. Compara manos.

    Parameters
    hole_cards       : cartas del jugador (formato treys)
    community_cards  : cartas comunitarias ya visibles
    n_simulations    : número de simulaciones

    Returns, float  equity ∈ [0, 1]  (probabilidad estimada de ganar)
    """
    if len(hole_cards) < 2:
        return 0.5

    used    = set(hole_cards + community_cards)
    deck    = [c for c in ALL_CARDS if c not in used]
    wins    = 0
    ties    = 0
    n_board = 5 - len(community_cards)

    for _ in range(n_simulations):
        sample = random.sample(deck, n_board + 2)
        board  = community_cards + sample[:n_board]
        opp    = sample[n_board:]

        try:
            my_score  = EVALUATOR.evaluate(board, hole_cards)
            opp_score = EVALUATOR.evaluate(board, opp)
        except Exception:
            continue

        if my_score < opp_score:        # menor score = mejor mano en treys
            wins += 1
        elif my_score == opp_score:
            ties += 1

    return (wins + 0.5 * ties) / n_simulations


print("Evaluación de manos definida.")

# Prueba rápida
test_hole = [Card.new('As'), Card.new('Kh')]
test_comm = [Card.new('Ac'), Card.new('Ad'), Card.new('2h')]
eq = simulate_equity(test_hole, test_comm, n_simulations=500)
hs = evaluate_hand(test_hole, test_comm)
print(f"  Mano: A♠ K♥ | Tablero: A♣ A♦ 2♥")
print(f"  Fuerza normalizada : {hs:.3f}")
print(f"  Equity estimada    : {eq:.3f}")


Evaluación de manos definida.
  Mano: A♠ K♥ | Tablero: A♣ A♦ 2♥
  Fuerza normalizada : 0.783
  Equity estimada    : 0.969


In [6]:
# 1.5  Simulador de partida

class BaseAgent: # Clase base para todos los agentes
    name: str = "BaseAgent"

    def choose_action(self, state: GameState) -> Action:
        raise NotImplementedError


class RandomAgent(BaseAgent): # Agente que elige acciones uniformemente al azar
    name = "Random"

    def choose_action(self, state: GameState) -> Action:
        return random.choice(get_legal_actions(state))


class CallAgent(BaseAgent): # Agente pasivo: siempre hace call/check; fold solo si es la única opción.
    name = "Call"

    def choose_action(self, state: GameState) -> Action:
        legal = get_legal_actions(state)
        for a in [Action.CALL, Action.CHECK]:
            if a in legal:
                return a
        return legal[0]


def deal_hand(rng: random.Random) -> Tuple[List[int], List[int], List[int]]:
    """
    Reparte una mano completa: 2 hole cards por jugador + 5 comunitarias.
    Returns (hole_p0, hole_p1, community)
    """
    deck = list(ALL_CARDS)
    rng.shuffle(deck)
    return deck[0:2], deck[2:4], deck[4:9]


def play_hand(
    agent0: BaseAgent,
    agent1: BaseAgent,
    rng: Optional[random.Random] = None,
) -> Tuple[int, float]:
    """
    Simula una mano completa entre dos agentes.

    Returns
    winner : int   (0 o 1; -1 si empate)
    profit : float fichas ganadas por el agente 0 (negativo = pierde)
    """
    if rng is None:
        rng = random.Random()

    hole0, hole1, community = deal_hand(rng)
    stacks = [float(STARTING_STACK), float(STARTING_STACK)]

    # Ciegas
    stacks[0] -= BIG_BLIND / 2    # small blind
    stacks[1] -= BIG_BLIND        # big blind
    pot = BIG_BLIND * 1.5
    current_bet = float(BIG_BLIND)

    agents        = [agent0, agent1]
    hole_cards    = [hole0, hole1]
    phases        = [Phase.PREFLOP, Phase.FLOP, Phase.TURN, Phase.RIVER]
    comm_revealed = [0, 3, 4, 5]
    folded        = [False, False]
    winner        = -1
    profit        = 0.0

    for phase_idx, (phase, n_comm) in enumerate(zip(phases, comm_revealed)):
        if any(folded) or any(s <= 0 for s in stacks):
            break

        comm_now = community[:n_comm]
        if phase_idx > 0:
            current_bet = 0.0    # nueva calle

        # Hasta dos acciones por calle (simplificado)
        for turn in range(2):
            actor = turn % 2
            if folded[actor] or stacks[actor] <= 0:
                continue

            state = GameState(
                hole_cards    = hole_cards[actor],
                community_cards = comm_now,
                pot           = pot,
                stacks        = list(stacks),
                current_bet   = current_bet,
                phase         = phase,
                acting_player = actor,
                position      = actor,
            )
            action = agents[actor].choose_action(state)
            bet_size = action_to_bet_size(action, state)

            if action == Action.FOLD:
                folded[actor] = True
                winner = 1 - actor
                profit = (STARTING_STACK - stacks[0]) * (-1 if actor == 0 else 1)
                break
            elif action in (Action.CHECK,):
                pass
            elif action == Action.CALL:
                paid = min(current_bet, stacks[actor])
                stacks[actor] -= paid
                pot += paid
            elif action in (Action.RAISE_HALF, Action.RAISE_POT, Action.RAISE_2POT, Action.ALL_IN):
                paid = min(bet_size, stacks[actor])
                stacks[actor] -= paid
                pot += paid
                current_bet = paid
        else:
            continue
        break

    # Showdown si nadie fold
    if not any(folded):
        score0 = EVALUATOR.evaluate(community[:5], hole0) if len(community) >= 5 else 7462
        score1 = EVALUATOR.evaluate(community[:5], hole1) if len(community) >= 5 else 7462
        if score0 < score1:
            winner = 0
        elif score1 < score0:
            winner = 1
        else:
            winner = -1

        if winner == 0:
            profit = pot - (STARTING_STACK - stacks[0])
        elif winner == 1:
            profit = -(STARTING_STACK - stacks[0])
        else:
            profit = 0.0

    return winner, profit


def play_tournament(
    agents: List[BaseAgent],
    n_hands: int = 500,
    seed: int = RANDOM_STATE,
) -> pd.DataFrame:
    """
    Torneo round-robin: cada agente juega n_hands manos contra cada otro.

    Returns
    pd.DataFrame con columnas [Agent, Opponent, Wins, Losses, Ties, Profit, WinRate]
    """
    rng     = random.Random(seed)
    records = []

    pairs = list(itertools.combinations(range(len(agents)), 2))
    for i, j in tqdm(pairs, desc="Torneo round-robin"):
        a0, a1 = agents[i], agents[j]
        wins0, wins1, ties0 = 0, 0, 0
        profit0 = 0.0

        for _ in range(n_hands):
            w, p = play_hand(a0, a1, rng)
            profit0 += p
            if w == 0:   wins0 += 1
            elif w == 1: wins1 += 1
            else:        ties0 += 1

        records.append({
            "Agent":    a0.name, "Opponent": a1.name,
            "Wins":     wins0,   "Losses":   wins1,   "Ties": ties0,
            "Profit":   profit0, "WinRate":  wins0 / n_hands,
        })
        records.append({
            "Agent":    a1.name, "Opponent": a0.name,
            "Wins":     wins1,   "Losses":   wins0,   "Ties": ties0,
            "Profit":   -profit0, "WinRate": wins1 / n_hands,
        })

    return pd.DataFrame(records)


print("Simulador de partida definido.")
print("  Prueba: Random vs Call (10 manos):")
rng_test = random.Random(42)
for _ in range(3):
    w, p = play_hand(RandomAgent(), CallAgent(), rng_test)
    print(f"    Ganador: {w} | Profit p0: {p:.0f}")


Simulador de partida definido.
  Prueba: Random vs Call (10 manos):
    Ganador: 1 | Profit p0: -10
    Ganador: 0 | Profit p0: 1000
    Ganador: 1 | Profit p0: -10


<a id='seccion-2'></a>
## Sección 2: Modelo 1 - Minimax con Poda Alpha-Beta

### Fundamento Teórico

**Minimax** es un algoritmo de búsqueda en árbol diseñado para juegos de **suma cero con dos jugadores**. El jugador A (MAX) trata de maximizar su utilidad; el jugador B (MIN) trata de minimizarla. El algoritmo explora el árbol de juego hasta cierta profundidad, calculando la utilidad de cada nodo hoja y propagándola hacia arriba:

- **Nodo MAX**: elige la acción de mayor utilidad
- **Nodo MIN**: elige la acción de menor utilidad
- **Nodo CHANCE**: promedia la utilidad sobre resultados equiprobables (cartas aleatorias)

#### Adaptación al Póker (Información Oculta)

El póker es un juego de **información imperfecta**: no conocemos las cartas del oponente. La solución es el **world sampling**: samplear N configuraciones posibles de cartas del oponente y promediar la utilidad del árbol sobre esas configuraciones.

#### Poda Alpha-Beta

Evita explorar sub-árboles que no pueden influir en la decisión final, reduciendo la complejidad de O(b^d) a O(b^(d/2)) en el caso promedio:
- **α**: mejor valor garantizado para MAX hasta ahora
- **β**: mejor valor garantizado para MIN hasta ahora
- Si β ≤ α, se poda el sub-árbol (no puede mejorar la decisión actual)

#### Función de Utilidad

Combina tres factores en las hojas:
$$U(s) = 0.6 \cdot \text{equity} + 0.2 \cdot \text{pot\_odds} + 0.2 \cdot \text{position\_bonus}$$


In [7]:
# 2.2 & 2.3  Minimax con Poda Alpha-Beta y World Sampling

class MinimaxAgent(BaseAgent):
    """
    Agente que usa Minimax con poda Alpha-Beta y world sampling
    para manejar la información oculta sobre las cartas del oponente.

    Parameters
    max_depth    : int   profundidad máxima del árbol (default 3)
    n_worlds     : int   número de mundos posibles a samplear (default 20)
    n_equity_sim : int   simulaciones Monte Carlo para equity
    """
    name = "Minimax"

    def __init__(
        self,
        max_depth:    int = 3,
        n_worlds:     int = 20,
        n_equity_sim: int = 200,
    ):
        self.max_depth    = max_depth
        self.n_worlds     = n_worlds
        self.n_equity_sim = n_equity_sim
        self._rng         = random.Random(RANDOM_STATE)

    # Función de utilidad en nodos hoja

    def _utility(self, state: GameState, opp_hole: List[int]) -> float:
        """
        Calcula la utilidad del estado terminal/hoja dado un mundo concreto.

        CORRECCIÓN 1 — Pot odds:
            Un pot_odds alto significa que el bote es grande en relación
            al costo de llamar → favorable para el jugador. Se usa
            directamente (sin invertir).

        CORRECCIÓN 2 — World sampling real:
            La utilidad ahora recibe las cartas del oponente explícitamente
            y compara fuerza de mano directamente, en lugar de usar Monte
            Carlo genérico que ignora las cartas del oponente.

        U = 0.6 * win_prob + 0.2 * pot_odds + 0.2 * position_bonus
        """
        # Comparación directa contra las cartas del oponente en este mundo
        comm = state.community_cards
        if len(comm) >= 3 and len(state.hole_cards) == 2 and len(opp_hole) == 2:
            try:
                my_score  = EVALUATOR.evaluate(comm, state.hole_cards)
                opp_score = EVALUATOR.evaluate(comm, opp_hole)
                # treys: menor score = mejor mano
                if my_score < opp_score:
                    win_prob = 1.0
                elif my_score == opp_score:
                    win_prob = 0.5
                else:
                    win_prob = 0.0
            except Exception:
                win_prob = 0.5
        else:
            # Preflop o sin suficientes cartas: estimación Monte Carlo
            win_prob = simulate_equity(
                state.hole_cards, comm, self.n_equity_sim
            ) if state.hole_cards else 0.5

        # Pot odds altos son FAVORABLES (el bote es grande vs. el costo de llamar)
        pot_odds_norm = state.pot_odds  # ← CORRECCIÓN: sin invertir

        # Ventaja de posición: actuar último (position=1) es mejor
        position_bonus = 0.6 if state.position == 1 else 0.4

        return 0.6 * win_prob + 0.2 * pot_odds_norm + 0.2 * position_bonus

    # Aplicar acción al estado

    def _apply_action(self, state: GameState, action: Action) -> GameState:
        """Retorna un nuevo GameState tras aplicar la acción."""
        new_state = deepcopy(state)
        actor     = state.acting_player
        bet_size  = action_to_bet_size(action, state)

        if action == Action.FOLD:
            new_state.phase = Phase.SHOWDOWN

        elif action in (Action.RAISE_HALF, Action.RAISE_POT, Action.RAISE_2POT, Action.ALL_IN):
            paid = min(bet_size, new_state.stacks[actor])
            new_state.stacks[actor] -= paid
            new_state.pot           += paid
            new_state.current_bet    = paid

        elif action == Action.CALL:
            paid = min(state.current_bet, new_state.stacks[actor])
            new_state.stacks[actor] -= paid
            new_state.pot           += paid
            new_state.current_bet    = 0.0
            # Avanzar fase si ambos igualaron
            phase_order = [Phase.PREFLOP, Phase.FLOP, Phase.TURN, Phase.RIVER, Phase.SHOWDOWN]
            idx = phase_order.index(new_state.phase)
            if idx < len(phase_order) - 1:
                new_state.phase = phase_order[idx + 1]

        new_state.acting_player = 1 - actor
        new_state.action_history.append((state.phase, actor, action))
        return new_state

    # Minimax recursivo

    def _minimax(
        self,
        state:    GameState,
        opp_hole: List[int],
        depth:    int,
        alpha:    float,
        beta:     float,
        is_max:   bool,
    ) -> float:
        """
        Algoritmo Minimax con poda Alpha-Beta.
        opp_hole: cartas del oponente fijadas para este mundo.

        Returns
        float  utilidad estimada del nodo
        """
        if depth == 0 or state.is_terminal:
            return self._utility(state, opp_hole)

        legal = get_legal_actions(state)

        if is_max:
            value = -float('inf')
            for action in legal:
                child = self._apply_action(state, action)
                value = max(value, self._minimax(child, opp_hole, depth - 1, alpha, beta, False))
                alpha = max(alpha, value)
                if beta <= alpha:
                    break   # poda beta
            return value
        else:
            value = float('inf')
            for action in legal:
                child = self._apply_action(state, action)
                value = min(value, self._minimax(child, opp_hole, depth - 1, alpha, beta, True))
                beta  = min(beta, value)
                if beta <= alpha:
                    break   # poda alpha
            return value

    # World Sampling (Better)

    def _sample_world(self, state: GameState) -> Tuple[GameState, List[int]]:
        """
        Samplea un mundo posible asignando cartas aleatorias al oponente.

        Ahora retorna las cartas del oponente explícitamente
        para que _utility pueda comparar manos directamente en lugar de
        ignorarlas. Esto hace que cada mundo sea genuinamente diferente.

        Returns
        (world_state, opp_hole_cards)
        """
        used     = set(state.hole_cards + state.community_cards)
        deck     = [c for c in ALL_CARDS if c not in used]
        opp_hole = self._rng.sample(deck, 2)
        world    = deepcopy(state)
        return world, opp_hole

    # Método principal

    def choose_action(self, state: GameState) -> Action:
        """
        Selecciona la mejor acción usando Minimax + world sampling.

        Para cada acción legal, promedia el valor Minimax sobre
        self.n_worlds configuraciones distintas de cartas del oponente.
        Cada mundo produce una evaluación genuinamente diferente porque
        las cartas del oponente se pasan hasta los nodos hoja.
        """
        legal = get_legal_actions(state)
        if len(legal) == 1:
            return legal[0]

        best_action = legal[0]
        best_value  = -float('inf')

        for action in legal:
            world_values = []
            for _ in range(self.n_worlds):
                world, opp_hole = self._sample_world(state)   # mundo distinto
                child = self._apply_action(world, action)
                v     = self._minimax(child, opp_hole, self.max_depth - 1,
                                      -np.inf, np.inf, False)
                world_values.append(v)

            avg_value = np.mean(world_values)
            if avg_value > best_value:
                best_value  = avg_value
                best_action = action

        return best_action


print("MinimaxAgent definido (world sampling y pot_odds corregidos).")


MinimaxAgent definido (world sampling y pot_odds corregidos).


In [8]:
# Sanity Check: Minimax
print("SANITY CHECK - Minimax")

# Mano muy fuerte: trío de ases
hole_mm   = [Card.new('As'), Card.new('Ah')]
comm_mm   = [Card.new('Ac'), Card.new('Kd'), Card.new('2h')]
state_mm  = GameState(
    hole_cards      = hole_mm,
    community_cards = comm_mm,
    pot             = 200,
    stacks          = [800, 800],
    current_bet     = 50,
    phase           = Phase.FLOP,
    acting_player   = 0,
    position        = 1,   # posición tardía (ventaja)
)

agent_mm = MinimaxAgent(max_depth=2, n_worlds=10, n_equity_sim=100)
t0       = time.time()
action   = agent_mm.choose_action(state_mm)
elapsed  = (time.time() - t0) * 1000

equity_mm = simulate_equity(hole_mm, comm_mm, 500)
print(f"  Mano       : {Card.ints_to_pretty_str(hole_mm)}")
print(f"  Tablero    : {Card.ints_to_pretty_str(comm_mm)}")
print(f"  Equity     : {equity_mm:.3f}")
print(f"  Pot        : 200 | Bet: 50 | Posición: LATE")
print(f"  Acción elegida: {ACTION_LABELS[action]}")
print(f"  Tiempo   : {elapsed:.1f} ms")
print()
print("  Razonamiento: con equity alta (trío de ases) y posición")
print("  ventajosa, Minimax debería preferir raise o all-in.")


SANITY CHECK - Minimax
  Mano       :  [A♠],[A♥] 
  Tablero    :  [A♣],[K♦],[2♥] 
  Equity     : 0.974
  Pot        : 200 | Bet: 50 | Posición: LATE
  Acción elegida: All-in
  Tiempo   : 6.7 ms

  Razonamiento: con equity alta (trío de ases) y posición
  ventajosa, Minimax debería preferir raise o all-in.


<a id='seccion-3'></a>
## Sección 3: Modelo 2 - Red Bayesiana

### Fundamento Teórico

El póker está dominado por la **incertidumbre**: no sabemos las cartas del oponente, su estilo de juego, ni cómo evolucionará el tablero. Las **Redes Bayesianas** modelan estas dependencias de forma explícita como un **grafo acíclico dirigido (DAG)** donde:

- Los **nodos** son variables aleatorias relevantes para la decisión
- Las **aristas** representan dependencias causales
- Cada nodo tiene una **Tabla de Probabilidad Condicional (CPT)** que cuantifica la relación

La inferencia usa el **Teorema de Bayes** para actualizar creencias dado el estado observado:

$$P(\text{BestAction} \mid \text{evidencia}) \propto P(\text{evidencia} \mid \text{BestAction}) \cdot P(\text{BestAction})$$

#### Ventajas
- Transparente: las decisiones son explicables por las CPTs
- Incorpora conocimiento experto de forma natural
- Actualización bayesiana a medida que se observa más información


In [9]:
# 3.2 & 3.3  Estructura y CPTs de la Red Bayesiana

def build_bayesian_network() -> Tuple[BayesianNetwork, VariableElimination]:
    """
    Construye la Red Bayesiana para decisiones de póker.

    Nodos:
        HandStrength     : WEAK(0) | MEDIUM(1) | STRONG(2) | VERY_STRONG(3)
        OpponentStrength : WEAK(0) | MEDIUM(1) | STRONG(2)
        Position         : EARLY(0) | LATE(1)
        PotOdds          : LOW(0)  | MEDIUM(1) | HIGH(2)
        OpponentTendency : PASSIVE(0) | AGGRESSIVE(1)
        BestAction       : FOLD(0) | CHECK_CALL(1) | RAISE(2) | ALL_IN(3)

    Returns
    (BayesianNetwork, VariableElimination)
    """
    #  Estructura del grafo 
    model = BayesianNetwork([
        ('HandStrength',     'BestAction'),
        ('OpponentStrength', 'BestAction'),
        ('Position',         'BestAction'),
        ('PotOdds',          'BestAction'),
        ('OpponentTendency', 'BestAction'),
        ('HandStrength',     'OpponentStrength'),  # prior del oponente
    ])

    # CPT: HandStrength (prior uniforme)
    cpd_hs = TabularCPD(
        variable='HandStrength', variable_card=4,
        values=[[0.30], [0.35], [0.25], [0.10]],
        state_names={'HandStrength': ['WEAK', 'MEDIUM', 'STRONG', 'VERY_STRONG']},
    )

    # CPT: OpponentStrength | HandStrength
    # Si tenemos mano fuerte, el oponente es probablemente más débil (selección adversa)
    cpd_os = TabularCPD(
        variable='OpponentStrength', variable_card=3,
        evidence=['HandStrength'], evidence_card=[4],
        values=[
            # WEAK  MED   STR   V_STR
            [0.20, 0.30, 0.40, 0.50],  # Oponente WEAK
            [0.40, 0.40, 0.40, 0.35],  # Oponente MEDIUM
            [0.40, 0.30, 0.20, 0.15],  # Oponente STRONG
        ],
        state_names={
            'OpponentStrength': ['WEAK', 'MEDIUM', 'STRONG'],
            'HandStrength':     ['WEAK', 'MEDIUM', 'STRONG', 'VERY_STRONG'],
        },
    )

    # CPT: Position (prior balanceado)
    cpd_pos = TabularCPD(
        variable='Position', variable_card=2,
        values=[[0.5], [0.5]],
        state_names={'Position': ['EARLY', 'LATE']},
    )

    # CPT: PotOdds (prior)
    cpd_po = TabularCPD(
        variable='PotOdds', variable_card=3,
        values=[[0.40], [0.40], [0.20]],
        state_names={'PotOdds': ['LOW', 'MEDIUM', 'HIGH']},
    )

    # CPT: OpponentTendency (prior) 
    cpd_ot = TabularCPD(
        variable='OpponentTendency', variable_card=2,
        values=[[0.55], [0.45]],
        state_names={'OpponentTendency': ['PASSIVE', 'AGGRESSIVE']},
    )

    # CPT: BestAction | HandStrength, OpponentStrength, Position, PotOdds, OpponentTendency 
    # Dimensiones: 4 × 3 × 2 × 3 × 2 = 144 combinaciones de padres
    # Para cada combinación definimos P(FOLD, CHECK/CALL, RAISE, ALL_IN)
    n_parents = 4 * 3 * 2 * 3 * 2   # 144

    fold_p       = np.zeros(n_parents)
    check_call_p = np.zeros(n_parents)
    raise_p      = np.zeros(n_parents)
    all_in_p     = np.zeros(n_parents)

    hs_vals  = ['WEAK', 'MEDIUM', 'STRONG', 'VERY_STRONG']
    os_vals  = ['WEAK', 'MEDIUM', 'STRONG']
    pos_vals = ['EARLY', 'LATE']
    po_vals  = ['LOW', 'MEDIUM', 'HIGH']
    ot_vals  = ['PASSIVE', 'AGGRESSIVE']

    idx = 0
    for hs in hs_vals:
        for os in os_vals:
            for pos in pos_vals:
                for po in po_vals:
                    for ot in ot_vals:
                        # Lógica de estrategia de póker
                        hs_score  = hs_vals.index(hs)   # 0-3
                        os_score  = os_vals.index(os)   # 0-2
                        late      = pos == 'LATE'
                        po_score  = po_vals.index(po)   # 0-2
                        aggressive= ot == 'AGGRESSIVE'

                        # Base según fuerza de mano
                        if hs_score == 0:    # WEAK
                            f, c, r, a = 0.55, 0.30, 0.12, 0.03
                        elif hs_score == 1:  # MEDIUM
                            f, c, r, a = 0.20, 0.45, 0.28, 0.07
                        elif hs_score == 2:  # STRONG
                            f, c, r, a = 0.05, 0.25, 0.50, 0.20
                        else:                # VERY_STRONG
                            f, c, r, a = 0.02, 0.10, 0.40, 0.48

                        # Ajuste por oponente fuerte → más cautela
                        if os_score == 2:
                            f += 0.10; c += 0.05; r -= 0.10; a -= 0.05

                        # Ajuste por posición tardía → más agresivo
                        if late:
                            f -= 0.05; r += 0.03; a += 0.02

                        # Ajuste por pot odds altos (CALL tiene sentido)
                        if po_score == 2:
                            c += 0.10; f -= 0.10

                        # Ajuste por oponente agresivo → más calls, menos raises
                        if aggressive:
                            c += 0.05; r -= 0.05

                        # Normalizar a distribución válida
                        vals = np.array([f, c, r, a])
                        vals = np.clip(vals, 0.01, 1.0)
                        vals /= vals.sum()

                        fold_p[idx], check_call_p[idx], raise_p[idx], all_in_p[idx] = vals
                        idx += 1

    cpd_ba = TabularCPD(
        variable='BestAction', variable_card=4,
        evidence=['HandStrength', 'OpponentStrength', 'Position', 'PotOdds', 'OpponentTendency'],
        evidence_card=[4, 3, 2, 3, 2],
        values=[fold_p, check_call_p, raise_p, all_in_p],
        state_names={
            'BestAction':       ['FOLD', 'CHECK_CALL', 'RAISE', 'ALL_IN'],
            'HandStrength':     hs_vals,
            'OpponentStrength': os_vals,
            'Position':         pos_vals,
            'PotOdds':          po_vals,
            'OpponentTendency': ot_vals,
        },
    )

    model.add_cpds(cpd_hs, cpd_os, cpd_pos, cpd_po, cpd_ot, cpd_ba)
    assert model.check_model(), "❌ El modelo bayesiano tiene errores de consistencia"
    inference = VariableElimination(model)

    return model, inference


bn_model, bn_inference = build_bayesian_network()
print("Red Bayesiana construida y validada.")


Red Bayesiana construida y validada.


In [10]:
# 3.4  Agente Bayesiano

def discretize_hand_strength(equity: float) -> str:
    """Convierte equity continua a categoría discreta."""
    if equity < 0.35:  return 'WEAK'
    if equity < 0.55:  return 'MEDIUM'
    if equity < 0.75:  return 'STRONG'
    return 'VERY_STRONG'

def discretize_pot_odds(pot_odds: float) -> str:
    """Discretiza el pot odds a LOW/MEDIUM/HIGH."""
    if pot_odds < 0.25:  return 'LOW'
    if pot_odds < 0.45:  return 'MEDIUM'
    return 'HIGH'

def infer_opponent_tendency(action_history) -> str:
    """Infiere si el oponente es agresivo basándose en su historial."""
    opp_raises = sum(
        1 for (_, player, action) in action_history
        if player == 1 and action in (
            Action.RAISE_HALF, Action.RAISE_POT,
            Action.RAISE_2POT, Action.ALL_IN
        )
    )
    opp_total = sum(1 for (_, player, _) in action_history if player == 1)
    if opp_total == 0:
        return 'PASSIVE'
    return 'AGGRESSIVE' if opp_raises / opp_total > 0.3 else 'PASSIVE'


# Mapeo de acción bayesiana → Action del juego
BAYESIAN_TO_ACTION = {
    'FOLD':       Action.FOLD,
    'CHECK_CALL': Action.CHECK,   # se ajustará según current_bet
    'RAISE':      Action.RAISE_POT,
    'ALL_IN':     Action.ALL_IN,
}


class BayesianAgent(BaseAgent):
    """
    Agente que usa la Red Bayesiana para tomar decisiones.

    Discretiza el estado continuo, construye la evidencia
    y consulta la red para obtener la distribución posterior
    sobre acciones. Elige la acción de mayor probabilidad posterior.
    """
    name = "Bayesian"

    def __init__(self, inference: VariableElimination = None):
        self._inf = inference if inference is not None else bn_inference
        self._rng = random.Random(RANDOM_STATE)

    def choose_action(self, state: GameState) -> Action:
        """Consulta la red y retorna la acción de mayor prob. posterior."""
        # 1. Calcular equity (rápido: 150 sims)
        equity = simulate_equity(
            state.hole_cards, state.community_cards, n_simulations=150
        ) if state.hole_cards else 0.5

        # 2. Discretizar evidencia
        hs  = discretize_hand_strength(equity)
        po  = discretize_pot_odds(state.pot_odds)
        pos = 'LATE' if state.position == 1 else 'EARLY'
        ot  = infer_opponent_tendency(state.action_history)

        evidence = {
            'HandStrength':     hs,
            'Position':         pos,
            'PotOdds':          po,
            'OpponentTendency': ot,
        }

        # 3. Inferencia bayesiana
        try:
            result     = self._inf.query(['BestAction'], evidence=evidence, show_progress=False)
            probs      = result.values
            categories = ['FOLD', 'CHECK_CALL', 'RAISE', 'ALL_IN']
            best_cat   = categories[np.argmax(probs)]
        except Exception:
            best_cat = 'CHECK_CALL'

        # 4. Convertir a acción legal
        action     = BAYESIAN_TO_ACTION[best_cat]
        legal      = get_legal_actions(state)

        # Ajustar CHECK ↔ CALL según la situación
        if action == Action.CHECK:
            if Action.CHECK not in legal and Action.CALL in legal:
                action = Action.CALL
            elif Action.CHECK not in legal and Action.CALL not in legal:
                action = Action.FOLD

        if action not in legal:
            # Fallback: acción más agresiva disponible
            for fallback in [Action.CALL, Action.CHECK, Action.FOLD]:
                if fallback in legal:
                    action = fallback
                    break

        return action


bayes_agent = BayesianAgent()
print("BayesianAgent definido.")


BayesianAgent definido.


In [11]:
# Sanity Check: Red Bayesiana
print("SANITY CHECK - Red Bayesiana")

# Escenario 1: Mano muy fuerte, posición tardía
hole_bn = [Card.new('Ks'), Card.new('Kh')]
comm_bn = [Card.new('Kd'), Card.new('2c'), Card.new('7s')]
state_bn = GameState(
    hole_cards=hole_bn, community_cards=comm_bn,
    pot=300, stacks=[700, 700], current_bet=0,
    phase=Phase.FLOP, acting_player=0, position=1,
)

t0 = time.time()
action_bn = bayes_agent.choose_action(state_bn)
elapsed_bn = (time.time() - t0) * 1000

eq_bn = simulate_equity(hole_bn, comm_bn, 500)
print(f"  Mano         : {Card.ints_to_pretty_str(hole_bn)}")
print(f"  Tablero      : {Card.ints_to_pretty_str(comm_bn)}")
print(f"  Equity       : {eq_bn:.3f}")
print(f"  HandStrength : {discretize_hand_strength(eq_bn)}")
print(f"  Posición     : LATE | Pot Odds: 0 (no hay apuesta)")
print(f"  Acción elegida: {ACTION_LABELS[action_bn]}")
print(f"  Tiempo     : {elapsed_bn:.1f} ms")
print()

# Escenario 2: Mano débil, oponente agresivo
hole_bn2 = [Card.new('2s'), Card.new('7h')]
comm_bn2 = [Card.new('Ac'), Card.new('Kh'), Card.new('Qd')]
state_bn2 = GameState(
    hole_cards=hole_bn2, community_cards=comm_bn2,
    pot=200, stacks=[500, 800], current_bet=150,
    phase=Phase.FLOP, acting_player=0, position=0,
)
# Simular historial con oponente agresivo
state_bn2.action_history = [
    (Phase.PREFLOP, 1, Action.RAISE_2POT),
    (Phase.FLOP,    1, Action.RAISE_POT),
]

action_bn2 = bayes_agent.choose_action(state_bn2)
eq_bn2 = simulate_equity(hole_bn2, comm_bn2, 300)
print(f"  Mano         : {Card.ints_to_pretty_str(hole_bn2)}")
print(f"  Tablero      : {Card.ints_to_pretty_str(comm_bn2)}")
print(f"  Equity       : {eq_bn2:.3f}")
print(f"  Oponente     : AGGRESSIVE (historial de raises)")
print(f"  Acción elegida: {ACTION_LABELS[action_bn2]}")
print()
print("  Razonamiento: con mano débil (2-7 en tablero AKQ),")
print("  oponente agresivo y pot odds altos → la red debería")
print("  sugerir FOLD.")


SANITY CHECK - Red Bayesiana
  Mano         :  [K♠],[K♥] 
  Tablero      :  [K♦],[2♣],[7♠] 
  Equity       : 0.986
  HandStrength : VERY_STRONG
  Posición     : LATE | Pot Odds: 0 (no hay apuesta)
  Acción elegida: All-in
  Tiempo     : 4.7 ms

  Mano         :  [2♠],[7♥] 
  Tablero      :  [A♣],[K♥],[Q♦] 
  Equity       : 0.212
  Oponente     : AGGRESSIVE (historial de raises)
  Acción elegida: Fold

  Razonamiento: con mano débil (2-7 en tablero AKQ),
  oponente agresivo y pot odds altos → la red debería
  sugerir FOLD.
